In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from ydata_profiling import ProfileReport 
from sklearn.decomposition import PCA
from sklearn.preprocessing import MinMaxScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from statsmodels.graphics.mosaicplot import mosaic


plt.rcParams['figure.figsize'] = [6,6]

In [ ]:
main_database = pd.read_csv("auto_buying_database.csv")
main_database.sort_values('Price')

# Pre-profiling

In [ ]:
prof = ProfileReport(main_database)
prof.to_file('profile.html')

# EDA and Data Engineering

In [ ]:
df1 = main_database.copy()
df1.head()

df1 = df1.replace({'No': 0, 'Yes': 1})
df1["Wife Working"] = df1["Wife Working"].replace({'m': 0})
df1 = df1.drop("Make",axis=1)
df1 = df1.drop("Salary",axis=1)
df1 = df1.drop("Wife Salary",axis=1)
df1 = df1.drop([12,15,54],axis=0)

scaler = MinMaxScaler()
df1[["Total Salary","Price"]] = scaler.fit_transform(df1[["Total Salary","Price"]])

df1.head()

In [ ]:
dfw=df1[['No of Dependents','Total Salary','Price']]
dfw.head()

prof2 = ProfileReport(dfw)
prof2.to_file('profile2.html')

In [ ]:
pca = PCA(n_components=3)
pca.fit(dfw)
index=[]
var_prop=pca.explained_variance_ratio_
stdev=np.sqrt(pca.explained_variance_)
cumul=[]
for i in range(0,3):
    index.append("PC" + str(i+1))
    cumul.append(sum(pca.explained_variance_ratio_[0:i+1]))

table1 = pd.DataFrame({"Proportion of Variance":var_prop,
                       "Standard Deviation":stdev,
                       "Cumulative Proportion":cumul
                       },index=index)    

table1

In [ ]:
index=dfw.columns.tolist()
table2 = pd.DataFrame({"PC1":pca.components_[0],
                       "PC2":pca.components_[1],
                       "PC3":pca.components_[2]
                       },index=index)    

table2

In [ ]:
segments=np.arange(2,20)
cluster_dist=[]


for i in segments:
    kmeans = KMeans(n_clusters=i, random_state=0, n_init=10).fit(dfw)
    cluster_dist.append((kmeans.inertia_))
    label = kmeans.labels_
    sil_coeff = silhouette_score(dfw, label, metric='euclidean')
    print("For n_clusters={}, The Silhouette Coefficient is {}".format(i, sil_coeff))

plt.figure(figsize=(15,10))
plt.plot(segments, cluster_dist)
plt.xlabel("number of segments")
plt.ylabel("sum of within cluster distances")

In [ ]:
kmeans = KMeans(n_clusters=6, random_state=0, n_init=10).fit(dfw)
df1['label'] = pd.Series(kmeans.labels_+1,index=df1.index)

In [ ]:
prof3 = ProfileReport(df1)
prof3.to_file('profile3.html')

In [ ]:
mosaic(df1,['label','Profession'])
plt.xlabel("Profession vs Segment")

In [ ]:
mosaic(df1,['label','Marrital Status'])
plt.xlabel("Marital Status vs Segment")

In [ ]:
mosaic(df1,['label','Education'],labelizer=lambda k: " ")
plt.xlabel("Education vs Segment")

In [ ]:
mosaic(df1,['label','No of Dependents'])
plt.xlabel("Dependent Count vs Segment")